# Paper 2 — Random Forest v2: Additional Covariates (LST, Road Distance, Land-Use)

**SCRIPT: Paper2_RF_v2_Additional_Covariates.ipynb**

Purpose: raise the Random Forest cross-validated R² above the current 0.554 by adding three
covariates identified in Discussion as missing from the v1 model:
1. **Land Surface Temperature (LST)** — MODIS MOD11A2, via Google Earth Engine
2. **Distance to nearest major road** — reuses `road_distance_osm_verified.csv` from the
   Paper 2 road-distance work (Section 2.7 in the original draft)
3. **Distance to nearest DIW facility** — computed from the same DIW registry already
   geocoded for Section 2.7 Industrial Source Geocoding (391 records, 384 geocoded)
4. **Land-use classification** — ESA WorldCover 10m, via Google Earth Engine

This notebook assumes you already have a working RF v1 training table (the one used to
produce Table 7 / Figure 4-5 in the manuscript). Fill in the `TODO` markers with your actual
file paths / column names before running.

**Also addresses reviewer concerns about reproducibility:**
- Explicit hyperparameters (`n_estimators`, `max_depth`, `min_samples_leaf`, `random_state`)
- Both **random 10-fold CV** (as in v1) and **spatial-block CV** (new) are reported side by
  side, since random CV on spatially autocorrelated data can be optimistic (a reviewer will
  ask about this — see Discussion §4).
- RMSE and MAE reported alongside R² for every fold.


In [ ]:
# SECTION: Setup & defensive imports
import sys, subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg in ["geopandas", "scikit-learn", "earthengine-api", "geemap"]:
    _ensure(_pkg.split("-")[0] if _pkg != "earthengine-api" else "ee")

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

RANDOM_STATE = 42  # fixed seed for reproducibility — report this in Methods


In [ ]:
# SECTION: Earth Engine auth (defensive — skip if already initialized)
# NOTE (2024+ change): Earth Engine now requires a linked Google Cloud Project even for
# non-commercial / research use. If you have not registered one yet, go to:
#   https://code.earthengine.google.com/register
# and choose "Unpaid usage" -> create/select a Cloud Project -> copy its Project ID below.

import ee

EE_PROJECT_ID = "REPLACE_WITH_YOUR_PROJECT_ID"  # TODO: e.g. "ee-songwuts65"

try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)

print("Earth Engine ready, project:", EE_PROJECT_ID)


## Step 1 — Load the existing RF v1 training table

TODO: replace with the actual upload/path used in your original RF notebook.
Expected columns (adjust names to match yours): `lat`, `lon`, `date`, `pm25`, `pm10`, `tsp`,
plus the 17 existing predictors (AOD, met variables, month, day_of_week, dist_to_fire,
dist_to_nearest_factory, dist_to_risk_factory, etc. — per Figure 4 feature list).


In [ ]:
# SECTION: Load RF v1 training data (defensive upload pattern, matches your existing convention)
from google.colab import files

def load_or_upload(default_name):
    import os
    if os.path.exists(default_name):
        return pd.read_csv(default_name)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file uploaded — expected {default_name}")
    fname = list(uploaded.keys())[0]
    return pd.read_csv(fname)

# TODO: set this to your actual RF v1 training table filename
df = load_or_upload("rf_v1_training_table.csv")
print(df.shape)
df.head()


## Step 2 — Retrieve LST (MODIS MOD11A2, 1 km, 8-day composite)

Matches each row's `lat`/`lon`/`date` to the nearest 8-day LST composite. Uses the daytime
LST band (`LST_Day_1km`), scaled per MODIS convention (`* 0.02 - 273.15` → °C).


In [ ]:
# SECTION: LST retrieval via Google Earth Engine
def get_lst_for_row(lat, lon, date_str, window_days=8):
    point = ee.Geometry.Point([lon, lat])
    date = ee.Date(date_str)
    coll = (ee.ImageCollection("MODIS/061/MOD11A2")
            .filterDate(date.advance(-window_days, "day"), date.advance(window_days, "day"))
            .select("LST_Day_1km"))
    img = coll.mean()
    val = img.reduceRegion(ee.Reducer.first(), point, 1000).get("LST_Day_1km")
    try:
        v = val.getInfo()
        return v * 0.02 - 273.15 if v is not None else np.nan
    except Exception:
        return np.nan

# NOTE: row-by-row GEE calls are slow for large tables (>5,000 rows).
# For the full 5,001-grid-point RF surface, batch via ee.FeatureCollection instead:

def get_lst_batch(points_df, date_str, window_days=8):
    """points_df needs columns lat, lon. Returns a Series aligned to points_df.index."""
    features = [
        ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {"idx": int(i)})
        for i, row in points_df.iterrows()
    ]
    fc = ee.FeatureCollection(features)
    date = ee.Date(date_str)
    img = (ee.ImageCollection("MODIS/061/MOD11A2")
           .filterDate(date.advance(-window_days, "day"), date.advance(window_days, "day"))
           .select("LST_Day_1km")
           .mean())
    sampled = img.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=1000).getInfo()
    out = pd.Series(index=points_df.index, dtype=float)
    for f in sampled["features"]:
        idx = f["properties"]["idx"]
        v = f["properties"].get("first")
        out.loc[idx] = (v * 0.02 - 273.15) if v is not None else np.nan
    return out

# Example usage per unique date in df (batches by date to limit GEE calls):
lst_values = pd.Series(index=df.index, dtype=float)
for date_str, group in df.groupby("date"):
    lst_values.loc[group.index] = get_lst_batch(group[["lat", "lon"]], str(date_str))

df["lst_c"] = lst_values
print("LST missing:", df["lst_c"].isna().mean().round(3))


## Step 3 — Distance to nearest major road

Reuses the OSM-verified road network already built for the Paper 2 road-distance work
(`road_distance_osm_verified.csv`). If that file has per-site distances only (not per RF
grid point), this recomputes distance for every row using the same road GeoDataFrame.


In [ ]:
# SECTION: Distance to nearest major road
# TODO: point this to the OSM road network file/graph already built for road_distance_osm_verified.csv
import osmnx as ox

# Option A — you already have a saved road GeoDataFrame/graph:
# roads_gdf = gpd.read_file("saraburi_major_roads.geojson")

# Option B — rebuild from OSM (bounding box should match your Paper 2 study area):
# TODO: confirm bbox matches the one used for road_distance_osm_verified.csv
north, south, east, west = 14.95, 14.25, 101.45, 100.55
roads_gdf = ox.graph_to_gdfs(
    ox.graph_from_bbox(north, south, east, west,
                        network_type="drive",
                        custom_filter='["highway"~"motorway|trunk|primary|secondary"]'),
    nodes=False, edges=True
)

points_gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
).to_crs("EPSG:32647")  # UTM 47N — same projection used in the consolidated spatial-stats notebook
roads_utm = roads_gdf.to_crs("EPSG:32647")

df["dist_to_road_km"] = points_gdf.geometry.apply(
    lambda p: roads_utm.distance(p).min() / 1000
)
print(df["dist_to_road_km"].describe())


## Step 4 — Distance to nearest DIW facility

Reuses the same 384 geocoded facilities from Section 2.7 (Industrial Source Geocoding) —
no new geocoding needed, just a nearest-neighbor distance calculation.


In [ ]:
# SECTION: Distance to nearest geocoded DIW facility
# TODO: path to the geocoded facility table used for Table 6 / Figure 3
facilities = load_or_upload("diw_facilities_geocoded.csv")  # needs lat, lon columns
fac_gdf = gpd.GeoDataFrame(
    facilities, geometry=gpd.points_from_xy(facilities.lon, facilities.lat), crs="EPSG:4326"
).to_crs("EPSG:32647")

df["dist_to_nearest_facility_km"] = points_gdf.geometry.apply(
    lambda p: fac_gdf.distance(p).min() / 1000
)
print(df["dist_to_nearest_facility_km"].describe())


## Step 5 — Land-use classification (ESA WorldCover, 10 m)

Extracts the dominant land-cover class in a 1 km buffer around each point (matching the RF
grid resolution), plus the built-up fraction specifically (often the most predictive class
for industrial/traffic PM2.5).


In [ ]:
# SECTION: Land-use via ESA WorldCover
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

def get_landuse_batch(points_df, buffer_m=500):
    features = [
        ee.Feature(ee.Geometry.Point([row.lon, row.lat]).buffer(buffer_m), {"idx": int(i)})
        for i, row in points_df.iterrows()
    ]
    fc = ee.FeatureCollection(features)
    # built-up class code in WorldCover v200 is 50
    builtup_mask = worldcover.eq(50)
    reduced = builtup_mask.reduceRegions(
        collection=fc, reducer=ee.Reducer.mean(), scale=10
    ).getInfo()
    out = pd.Series(index=points_df.index, dtype=float)
    for f in reduced["features"]:
        idx = f["properties"]["idx"]
        out.loc[idx] = f["properties"].get("mean", np.nan)
    return out

# Batch by unique coordinates to avoid redundant GEE calls (land-use doesn't vary by date)
unique_coords = df[["lat", "lon"]].drop_duplicates()
builtup_frac = get_landuse_batch(unique_coords)
unique_coords = unique_coords.assign(builtup_frac=builtup_frac.values)
df = df.merge(unique_coords, on=["lat", "lon"], how="left")
print(df["builtup_frac"].describe())


## Step 6 — Merge, clean, and re-train the Random Forest

Reports **both** random 10-fold CV (as in v1, for direct comparison) and spatial-block CV
(grouping by rounded lat/lon into ~2 km blocks) so the manuscript can report whether the
random-CV R² was optimistic due to spatial autocorrelation — a point the reviewers
specifically flagged.


In [ ]:
# SECTION: Final feature set and missing-data handling
new_features = ["lst_c", "dist_to_road_km", "dist_to_nearest_facility_km", "builtup_frac"]

# TODO: confirm this matches the exact 17 predictor names used in Figure 4 / Table 7
existing_features = [
    "month", "dist_to_nearest_fire_km", "wind_speed_10m", "relative_humidity",
    "wind_speed_50m", "aod_047", "aod_055", "rainfall", "temperature",
    "fire_count", "wind_direction", "ndvi", "surface_pressure", "evi",
    "dist_to_risk_factory_km", "dist_to_nearest_factory_km", "day_of_week",
]

feature_cols = existing_features + new_features
target_col = "pm25"  # TODO: repeat for pm10, tsp as separate models if needed

before_n = len(df)
df_model = df.dropna(subset=feature_cols + [target_col]).copy()
print(f"Dropped {before_n - len(df_model)} rows with missing values "
      f"({(before_n - len(df_model)) / before_n:.1%}) — report this in Methods.")

X = df_model[feature_cols]
y = df_model[target_col]


In [ ]:
# SECTION: Random Forest — explicit hyperparameters for reproducibility
rf_params = dict(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf = RandomForestRegressor(**rf_params)

scoring = {
    "r2": "r2",
    "rmse": lambda est, Xv, yv: -mean_squared_error(yv, est.predict(Xv), squared=False),
    "mae": lambda est, Xv, yv: -mean_absolute_error(yv, est.predict(Xv)),
}

# --- Random 10-fold CV (matches v1 methodology, for direct before/after comparison) ---
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_random = cross_validate(rf, X, y, cv=kf, scoring=scoring, return_train_score=False)

print("=== Random 10-fold CV (v2, with new covariates) ===")
print(f"Mean R²:   {cv_random['test_r2'].mean():.3f}  (best {cv_random['test_r2'].max():.3f}, "
      f"worst {cv_random['test_r2'].min():.3f})")
print(f"Mean RMSE: {-cv_random['test_rmse'].mean():.2f} \u00b5g/m\u00b3")
print(f"Mean MAE:  {-cv_random['test_mae'].mean():.2f} \u00b5g/m\u00b3")


In [ ]:
# SECTION: Spatial-block CV (addresses reviewer concern re: spatial autocorrelation leakage)
# Groups points into ~2 km blocks so nearby points never split across train/test
block_size_km = 2.0
df_model["block_x"] = (df_model["lon"] * 111 / block_size_km).round().astype(int)
df_model["block_y"] = (df_model["lat"] * 111 / block_size_km).round().astype(int)
df_model["block_id"] = df_model["block_x"].astype(str) + "_" + df_model["block_y"].astype(str)

n_blocks = df_model["block_id"].nunique()
n_splits_spatial = min(10, n_blocks)
gkf = GroupKFold(n_splits=n_splits_spatial)

cv_spatial = cross_validate(
    rf, X, y, cv=gkf, groups=df_model["block_id"], scoring=scoring, return_train_score=False
)

print(f"=== Spatial-block CV ({n_splits_spatial}-fold, ~{block_size_km} km blocks) ===")
print(f"Mean R²:   {cv_spatial['test_r2'].mean():.3f}  (best {cv_spatial['test_r2'].max():.3f}, "
      f"worst {cv_spatial['test_r2'].min():.3f})")
print(f"Mean RMSE: {-cv_spatial['test_rmse'].mean():.2f} \u00b5g/m\u00b3")
print(f"Mean MAE:  {-cv_spatial['test_mae'].mean():.2f} \u00b5g/m\u00b3")

print()
print("If spatial-block R² is notably lower than random-CV R², report BOTH in the revised "
      "Table 7 and note in Discussion that random CV was optimistic due to spatial "
      "autocorrelation — this pre-empts the exact question reviewers are likely to ask.")


## Step 7 — Feature importance (updated Figure 4)

Re-run the feature-importance ranking with the new covariates included, to see whether
LST / road-distance / land-use displace any of the original top-3 predictors
(month, distance-to-fire, wind speed).


In [ ]:
# SECTION: Refit on full data for feature importance + updated Figure 4
rf_full = RandomForestRegressor(**rf_params)
rf_full.fit(X, y)

importances = pd.Series(rf_full.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(7, 6))
colors = ["#d95f02" if f in importances.index[-3:] else "#1f78b4" for f in importances.index]
plt.barh(importances.index, importances.values, color=colors)
plt.xlabel("Relative importance")
plt.title("Random Forest feature importance (v2, with LST/road/land-use)")
plt.tight_layout()
plt.savefig("figure4_v2_feature_importance.png", dpi=200)
plt.show()

print(importances.sort_values(ascending=False))


## Step 8 — Export updated results for the manuscript

Produces the numbers to drop directly into a revised Table 7 and Section 3.5 text.


In [ ]:
# SECTION: Export summary for manuscript update (defensive download)
summary = pd.DataFrame({
    "metric": ["mean_r2_random_cv", "mean_r2_spatial_cv", "mean_rmse_random_cv",
               "mean_mae_random_cv", "n_estimators", "max_depth", "random_state",
               "n_features", "n_rows_used", "pct_rows_dropped_missing"],
    "value": [
        cv_random["test_r2"].mean(), cv_spatial["test_r2"].mean(),
        -cv_random["test_rmse"].mean(), -cv_random["test_mae"].mean(),
        rf_params["n_estimators"], str(rf_params["max_depth"]), rf_params["random_state"],
        len(feature_cols), len(df_model), (before_n - len(df_model)) / before_n,
    ]
})
summary.to_csv("rf_v2_summary_for_table7.csv", index=False)
print(summary)

try:
    files.download("rf_v2_summary_for_table7.csv")
    files.download("figure4_v2_feature_importance.png")
except Exception as e:
    print("Download skipped (not in Colab):", e)
